In [ ]:
%%bash
set -uo pipefail
echo "=== setup START $(date -u +%H:%M:%S) ==="
nvidia-smi --query-gpu=name,memory.total --format=csv
python -c "import tensorflow as tf; print('TF', tf.__version__, tf.config.list_physical_devices('GPU'))"
pip -q install gdown
mkdir -p /tmp/wv3/data /tmp/final
cd /tmp/final && gdown -q 1gOxjhlRD_Gz0_HqRUek35B2W4nDSsdbZ -O normalized_images.tar
cd /tmp/wv3/data && gdown -q 1BQzwi5d61_952hvhX1j07jldtrxTfJCG -O extra_images.tar \
  && gdown -q 1czdFXXsA4pB0Pjs37lgmXAbJOPk8Lnrp -O splits_v3.csv \
  && gdown -q 1X6aHEjKRkEZa46e9CSpelXnOPT4we05o -O all_images_before_dedup.csv
ls -l /tmp/final /tmp/wv3/data
cd /kaggle/working && rm -rf WasteVision && git clone -q https://github.com/ayushdhasmana433-rgb/WasteVision
git -C /kaggle/working/WasteVision log -1 --format='commit %h %s'
cd /kaggle/working/WasteVision/pipeline
printf "B2R123\nB3R123\nS2R123\nS3R123\nS4R123\n" > /tmp/queue_r123.txt
python -u run.py --drive-root /tmp/wv3 --data-root /tmp/data --tar /tmp/final/normalized_images.tar --extract-only
echo "=== queue START $(date -u +%H:%M:%S) ==="
python -u run.py --drive-root /tmp/wv3 --data-root /tmp/data --tar /tmp/final/normalized_images.tar --queue /tmp/queue_r123.txt 2>&1 | grep -v "meta_optimizer"
echo "=== queue END $(date -u +%H:%M:%S) ==="


In [ ]:
%%bash
mkdir -p /kaggle/working/results_r123
cp -v /tmp/wv3/results/*R123* /kaggle/working/results_r123/ 2>/dev/null
cd /kaggle/working && zip -qr results_r123.zip results_r123 && rm -rf WasteVision && ls -l /kaggle/working /kaggle/working/results_r123
